# 🧠 Module 03: Deep Learning Model Architectures
### Custom Scratch CNN & 4 Transfer Learning Networks (ResNet18, MobileNetV2, DenseNet121, VGG16)

This notebook implements:
1. `CustomOCTCNN`: 4-stage convolutional neural network built from scratch in PyTorch.
2. Transfer learning model builders with custom 4-class classification heads:
   - **ResNet18** (Residual skip connections)
   - **MobileNetV2** (Inverted residual bottlenecks for mobile efficiency)
   - **DenseNet121** (Dense feature reuse)
   - **VGG16** (Stacked 3x3 convolutions)
3. Model parameter inspection and tensor shape verification.

In [ ]:
import torch
import torch.nn as nn
from torchvision import models
import pandas as pd

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Compute Device: {device}")

### 3.1 CustomOCTCNN: Built From Scratch in PyTorch

In [ ]:
class CustomOCTCNN(nn.Module):
    """Custom Convolutional Neural Network built from scratch using PyTorch."""
    def __init__(self, num_classes=4, dropout_rate=0.4):
        super(CustomOCTCNN, self).__init__()
        
        self.features = nn.Sequential(
            # Stage 1: 3 -> 32 channels, 224x224 -> 112x112
            nn.Conv2d(3, 32, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(32),
            nn.ReLU(inplace=False),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            # Stage 2: 32 -> 64 channels, 112x112 -> 56x56
            nn.Conv2d(32, 64, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=False),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            # Stage 3: 64 -> 128 channels, 56x56 -> 28x28
            nn.Conv2d(64, 128, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=False),
            nn.MaxPool2d(kernel_size=2, stride=2),
            
            # Stage 4: 128 -> 256 channels, 28x28 -> 14x14
            nn.Conv2d(128, 256, kernel_size=3, stride=1, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=False),
            nn.AdaptiveAvgPool2d((4, 4))
        )
        
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(256 * 4 * 4, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(inplace=False),
            nn.Dropout(dropout_rate),
            nn.Linear(256, 64),
            nn.ReLU(inplace=False),
            nn.Dropout(dropout_rate * 0.75),
            nn.Linear(64, num_classes)
        )

    def forward(self, x):
        feat = self.features(x)
        out = self.classifier(feat)
        return out

print("CustomOCTCNN defined successfully.")

### 3.2 Pretrained Transfer Learning Model Builders

In [ ]:
def build_resnet18(num_classes=4, pretrained=True, freeze_backbone=False):
    weights = models.ResNet18_Weights.DEFAULT if pretrained else None
    model = models.resnet18(weights=weights)
    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False
    in_features = model.fc.in_features
    model.fc = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(in_features, num_classes)
    )
    return model


def build_vgg16(num_classes=4, pretrained=True, freeze_backbone=False):
    weights = models.VGG16_Weights.DEFAULT if pretrained else None
    model = models.vgg16(weights=weights)
    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False
    in_features = model.classifier[6].in_features
    model.classifier[6] = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(in_features, num_classes)
    )
    return model


def build_mobilenet_v2(num_classes=4, pretrained=True, freeze_backbone=False):
    weights = models.MobileNet_V2_Weights.DEFAULT if pretrained else None
    model = models.mobilenet_v2(weights=weights)
    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(in_features, num_classes)
    )
    return model


def build_densenet121(num_classes=4, pretrained=True, freeze_backbone=False):
    weights = models.DenseNet121_Weights.DEFAULT if pretrained else None
    model = models.densenet121(weights=weights)
    if freeze_backbone:
        for param in model.parameters():
            param.requires_grad = False
    in_features = model.classifier.in_features
    model.classifier = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(in_features, num_classes)
    )
    return model


SUPPORTED_MODELS = {
    "custom_cnn": CustomOCTCNN,
    "resnet18": build_resnet18,
    "mobilenet_v2": build_mobilenet_v2,
    "densenet121": build_densenet121,
    "vgg16": build_vgg16,
}


def build_model(model_name="resnet18", num_classes=4, pretrained=True, freeze_backbone=False):
    name = model_name.lower()
    if name == "custom_cnn":
        return CustomOCTCNN(num_classes=num_classes)
    elif name in SUPPORTED_MODELS:
        builder = SUPPORTED_MODELS[name]
        return builder(num_classes=num_classes, pretrained=pretrained, freeze_backbone=freeze_backbone)
    else:
        raise ValueError(f"Unknown model name: {model_name}. Supported: {list(SUPPORTED_MODELS.keys())}")

### 3.3 Model Parameter Analysis & Forward Pass Verification

In [ ]:
param_records = []
dummy_batch = torch.randn(2, 3, 224, 224)

for m_name in SUPPORTED_MODELS.keys():
    m = build_model(m_name, num_classes=4, pretrained=False)
    m.eval()
    with torch.no_grad():
        out = m(dummy_batch)
    
    total = sum(p.numel() for p in m.parameters())
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)
    param_records.append({
        "Model": m_name.upper(),
        "Total Params": f"{total:,}",
        "Trainable Params": f"{trainable:,}",
        "Output Shape": str(list(out.shape)),
        "Size (MB)": f"{total * 4 / (1024**2):.2f} MB"
    })

df_summary = pd.DataFrame(param_records)
print("Model Architecture Summary:")
print(df_summary.to_string(index=False))